Clean Silver - Analysis Dataset
Broader silver dataset for analysis/fraud detection — keeps Sale (Shipment + Return), Purchase, and Transfer entries, plus customer and lot information. Separate from the forecasting silver.

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_bronze, save_silver
from src.transform.battery.clean_silver import clean_to_silver_analysis
import pandas as pd
import datetime

blob_service = get_blob_service(storage_account_name, storage_account_key)

Read and trim

In [0]:
bronze = read_bronze(blob_service, "live/battery/battery_full_history.json")
bronze["postingDate"] = pd.to_datetime(bronze["postingDate"])

today = pd.Timestamp(datetime.date.today())
bronze_trimmed = bronze[bronze["postingDate"] < today].copy()

print(f"Bronze: {len(bronze)} rows -> trimmed: {len(bronze_trimmed)} rows")

Clean and save

In [0]:
analysis_silver = clean_to_silver_analysis(bronze_trimmed)
print(f"Analysis silver: {analysis_silver.shape}")

save_silver(blob_service, analysis_silver, "live/battery/battery_analysis_clean_live.json")
print("Saved to silver/live/battery/battery_analysis_clean_live.json")

Sanity checks

In [0]:
print(analysis_silver["entryType"].value_counts())
print(f"\nLotNo populated: {analysis_silver['lot_no'].notna().sum()} of {len(analysis_silver)}")

sale_rows = analysis_silver[analysis_silver["entryType"] == "Sale"]
print(f"\nIdentifiable customers (Sale rows): {sale_rows['is_identifiable_customer'].sum()} of {len(sale_rows)}")
print(f"Unique resolved customer identities: {sale_rows[sale_rows['is_identifiable_customer']]['resolved_customer_no'].nunique()}")

print("\nSample identifiable customers:")
print(sale_rows[sale_rows["is_identifiable_customer"]][["resolved_customer_no", "resolved_customer_name", "resolved_customer_address"]].drop_duplicates().head(10))

In [0]:
print(sale_rows[~sale_rows["is_identifiable_customer"] == False][["resolved_customer_name", "resolved_customer_address"]].head(10))

In [0]:
# Check main-customer fallback rows specifically (is_identifiable_customer == False)
fallback_check = analysis_silver[
    (analysis_silver["entryType"] == "Sale") & (analysis_silver["is_identifiable_customer"] == False)
]
print(fallback_check[["resolved_customer_name", "resolved_customer_address"]].head(10))

In [0]:
# Compare against raw bronze for the same kind of rows (no sub-customer at all)
print(bronze_trimmed[bronze_trimmed["subCustomerName"].astype(str).str.strip() == ""][
    ["customerName", "customerAddress", "customerAddress2", "customerCity"]
].head(10))

In [0]:
real_company_check = analysis_silver[
    (analysis_silver["entryType"] == "Sale") & (analysis_silver["resolved_customer_name"] == "SENSOR LANKA TRADING (PVT) LTD")
]
print(real_company_check[["resolved_customer_name", "resolved_customer_address"]].drop_duplicates())